In [1]:
import json

with open("homes.json", "r", encoding="utf-8") as f:
    input_data = json.load(f)

data = []

for elem in input_data:
    for i, turn in enumerate(elem["turns"]):
        if turn["speaker"] == "SYSTEM":
            data.append((elem["turns"][i-1]["utterance"], turn["utterance"]))

In [2]:
from transformers import BartForConditionalGeneration, BartTokenizer
import torch

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

for param in model.get_encoder().parameters():
    param.requires_grad = False

model.train()

task = "home_rental"

input_ids = []
attention_masks = []
labels = []

for input_text, target_text in data:
    enc = tokenizer(input_text, return_tensors="pt", padding=True)
    label_enc = tokenizer(target_text, return_tensors="pt", padding=True)

    input_ids.append(enc["input_ids"].squeeze(0))
    attention_masks.append(enc["attention_mask"].squeeze(0))

    label_ids = label_enc["input_ids"].squeeze(0)
    label_ids[label_ids == tokenizer.pad_token_id] = -100
    labels.append(label_ids)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(10):
    total_loss = 0
    for i in range(len(data)):
        output = model(
            input_ids=input_ids[i].unsqueeze(0),
            attention_mask=attention_masks[i].unsqueeze(0),
            labels=labels[i].unsqueeze(0)
        )
        loss = output.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1} done!")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Epoch 1 done!
Epoch 2 done!
Epoch 3 done!
Epoch 4 done!
Epoch 5 done!
Epoch 6 done!
Epoch 7 done!
Epoch 8 done!
Epoch 9 done!
Epoch 10 done!


In [3]:
model.save_pretrained("my_home_rental_model")
tokenizer.save_pretrained("my_home_rental_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_home_rental_model\\tokenizer_config.json',
 'my_home_rental_model\\tokenizer.json')

In [14]:
def generate_response(input_text):
    encoded_input = tokenizer(input_text, return_tensors="pt")
    output = model.generate(encoded_input["input_ids"], max_length=50, num_beams=4, early_stopping=True)
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

model.eval()

generate_response("I'm looking for a place to rent.")

'What area are you interested in?'

In [15]:
generate_response("I'd like to find one in San Francisco.")

'How many bedrooms do you want the apartment to have?'

In [16]:
generate_response("I need 2 bedrooms.")

'What area do you want the apartment to be in?'

In [17]:
generate_response("San Francisco.")

'How many bed rooms?'

In [18]:
generate_response("2")

'Do you need help with anything else?'

In [7]:
generate_response("Are pets allowed there?")

'Unfortunately pets are not allowed in the apartment.'

In [8]:
generate_response("It's okay. I'd like to schedule a visit.")

'When would you like to visit?'

In [9]:
generate_response("I'd like to visit on next Monday.")

"So you'd like to schedule a visit to Ann Darling Apartments next Monday?"

In [10]:
generate_response("Yes, that's correct.")

"It's been scheduled."

In [11]:
generate_response("Thank you very much.")

'Can I be of further assistance?'

In [12]:
generate_response("No, that's all. Thanks!")

'Have a great day.'